In [ ]:
from dotenv import load_dotenv

from app.agent.graph import DEFAULT_RECURSION_LIMIT
from app.config.settings import Settings
from app.runtime import build_agent_application
from langchain_core.messages import HumanMessage

print("Loading configuration and building the agent...")
load_dotenv()
settings = Settings.from_env(require_groq=True, require_tavily=True)
app = build_agent_application(settings)
print("Agent is ready.")


In [ ]:
# The graph state, nodes, tool routing, and graph assembly live in app/agent/.
print("Workflow assembled by app.runtime and app.agent.graph.")


In [ ]:
# Execução com Dados Falsos (Foco em Hipertrofia)

# ----------------------------------------------------
#  DADOS FIXOS DE TESTE 
# ----------------------------------------------------
USER_DATA_FAKE = {
    "age": 28,             
    "weight": 60.0,        
    "height_cm": 164.0,    
    "frequency": 5,       
    "gender": "female",    
    "goal": "hipertrofia",
    "name": "Amanda"       
}
# ----------------------------------------------------

def create_initial_prompt(user_data):
    """Cria o prompt inicial que FORÇA o fluxo de trabalho obrigatório e a saída personalizada."""
    
    total_workouts = user_data['frequency'] * 3
    
    return f"""
    Aja como um personal trainer de IA especialista.
    Sua tarefa é criar um plano de treino de musculação COMPLETO e DETALHADO.

    **REQUISITO CHAVE - ALTA PRIORIDADE:** O plano deve conter **EXATAMENTE {total_workouts} treinos distintos**, organizados em 3 semanas.
    
    **ESTRUTURA DE SAÍDA FINAL (USE EXATAMENTE ESTA ORDEM E FORMATO):**
    
    1.  **SEÇÃO DE TREINOS:** Gere os {total_workouts} treinos, enumerando-os de forma sequencial (Treino 1, Treino 2, ... até o Treino {total_workouts}), organizados em blocos de Semana 1, Semana 2 e Semana 3. Use séries/repetições e exercícios apropriados (NUNCA INVENTE NOMES).
    2.  **SEÇÃO DE ANÁLISE E DICAS:** Após a lista completa de treinos, crie uma seção final chamada "Análise do Coach e Estratégia".
        * Nesta seção, use os **Cálculos** (IMC/TMB) e o **Contexto RAG** para dar conselhos personalizados sobre:
            * **Meta Calórica** (Se precisa de superávit ou déficit).
            * **Progressão** (Como aumentar a carga nas Semanas 2 e 3).
            * **Foco Técnico** (1 ou 2 dicas de execução cruciais, Ex: Retração escapular no Supino).
    
    **Informações do Usuário:**
    - Objetivo: {user_data['goal']} | Frequência: {user_data['frequency']} dias/semana
    - Idade: {user_data['age']} | Peso: {user_data['weight']} | Altura: {user_data['height_cm']} | Gênero: {user_data['gender']}

    **Processo Obrigatório (Para o LangGraph):**
    1.  Execute CADA UMA das 3 ferramentas.
    2.  Após as ferramentas, PARE e gere a **ESTRUTURA DE SAÍDA FINAL** completa.
    """

# --- INÍCIO DA EXECUÇÃO ---

# 1. Prepara o prompt com os dados fixos
prompt = create_initial_prompt(USER_DATA_FAKE)
messages = [HumanMessage(content=prompt)]

print("\n--- INICIANDO AGENTE AI FITNESS COACH ---")
print(f" Perfil Fictício: {USER_DATA_FAKE['goal']} | Frequência: {USER_DATA_FAKE['frequency']} dias/semana")
print("-----------------------------------------")

# 2. Configura a execução
config = {"recursion_limit": DEFAULT_RECURSION_LIMIT}
final_plan_content = ""

# 3. Loop de Execução
for event in app.stream({"messages": messages}, config=config):
    for key, value in event.items():
        if key == "agent":
            agent_response = value['messages'][-1]
            if agent_response.tool_calls:
                print(f" CHAMA FERRAMENTA(S): {[tc['name'] for tc in agent_response.tool_calls]}") 
            else:
                final_plan_content = agent_response.content
                print(" GERAÇÃO FINAL DO PLANO.")
        elif key == "call_tools":
            tool_messages = value['messages']
            for tool_message in tool_messages:
                print(f"Resultado de {tool_message.name}: Sucesso. Dados inseridos no contexto.")

# 4. Saída Final
print("\n" + "="*70)
print(f"PLANO DE TREINO FINAL GERADO):")
print("="*70)

with open("plano_de_treino_final.md", "w", encoding="utf-8") as f:
    f.write(final_plan_content)
print("\n O plano também foi salvo em 'plano_de_treino_final.md'")